In [28]:
import pandas as pd

In [29]:
df_train=pd.read_csv("train.csv")
df_train.head(10)

,sample_id,catalog_content,image_link,price
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49
5,9259,"Item Name: Member's Mark Member's Mark, Basil,...",https://m.media-amazon.com/images/I/81nw0HXpCR...,18.50
6,191846,Item Name: Goya Foods Sazonador Total Seasonin...,https://m.media-amazon.com/images/I/61dH2Ebkt0...,5.99
7,222007,Item Name: VineCo Original Series Chilean Sauv...,https://m.media-amazon.com/images/I/71JllaFpxM...,94.00
8,37614,Item Name: NATURES PATH CEREAL FLK MULTIGRAIN ...,https://m.media-amazon.com/images/I/21O9RftI2v...,35.74
9,238044,Item Name: Mrs. Miller's Seedless Black Raspbe...,https://m.media-amazon.com/images/I/41miQk+RkJ...,31.80


In [30]:
df_test=pd.read_csv("test.csv")
df_test.head(10)

,sample_id,catalog_content,image_link
0,100179,Item Name: Rani 14-Spice Eshamaya's Mango Chut...,https://m.media-amazon.com/images/I/71hoAn78AW...
1,245611,Item Name: Natural MILK TEA Flavoring extract ...,https://m.media-amazon.com/images/I/61ex8NHCIj...
2,146263,Item Name: Honey Filled Hard Candy - Bulk Pack...,https://m.media-amazon.com/images/I/61KCM61J8e...
3,95658,Item Name: Vlasic Snack'mm's Kosher Dill 16 Oz...,https://m.media-amazon.com/images/I/51Ex6uOH7y...
4,36806,"Item Name: McCormick Culinary Vanilla Extract,...",https://m.media-amazon.com/images/I/71QYlrOMoS...
5,148239,"Item Name: Snyder's of Hanover Mini Pretzel, 1...",https://m.media-amazon.com/images/I/81ONUU3OAf...
6,92659,Item Name: Oregon Plum Purple 15 oz (Pack of 3...,https://m.media-amazon.com/images/I/519Plng8cp...
7,3780,"Item Name: Barkman Honey 059640 Honey, Clover,...",https://m.media-amazon.com/images/I/6134TQBo+v...
8,196940,Item Name: Against The Grain Gluten Free Origi...,https://m.media-amazon.com/images/I/41bK3vFt5R...
9,20472,"Item Name: Nature Valley Granola Bars, Sweet a...",https://m.media-amazon.com/images/I/81NWfSFYXB...


In [31]:
df_train.head()

,sample_id,catalog_content,image_link,price
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49


In [34]:
def parse_catalog_content(text):
    import re  # just in case
    import pandas as pd

    if not isinstance(text, str) or text.strip() == "":
        return pd.Series(["other", "other", "other", "other"])

    # Item Name
    item_name = re.search(r'Item Name:\s*(.*?)(?=Bullet Point 1:|Product Description:|Value:|$)', text, re.S)
    item_name = item_name.group(1).strip() if item_name else "other"

    # Bullet Points
    bullets = []
    for i in range(1,6):
        bp = re.search(r'Bullet Point {}:\s*(.*?)(?=Bullet Point {}:|Product Description:|Value:|$)'.format(i, i+1), text, re.S)
        if bp:
            bullets.append(bp.group(1).strip())

    # Product Description
    pd_text = re.search(r'Product Description:\s*(.*?)(?=Value:|$)', text, re.S)
    pd_text = pd_text.group(1).strip() if pd_text else ""

    # Combine description
    description = " ".join(bullets + [pd_text]) if (bullets + [pd_text]) else "other"

    # Value
    value = re.search(r'Value:\s*([\d\.]+)', text)
    value = value.group(1).strip() if value else "other"

    # Unit
    unit = re.search(r'Unit:\s*([A-Za-z ]+)', text)
    unit = unit.group(1).strip() if unit else "other"

    return pd.Series([item_name, description, value, unit])


In [35]:
df_train[['item_name','description','value','unit']] = df_train['catalog_content'].apply(lambda x: parse_catalog_content(x))
df_test[['item_name','description','value','unit']] = df_test['catalog_content'].apply(lambda x: parse_catalog_content(x))


In [37]:
df_train.head(10)

,sample_id,catalog_content,image_link,price,item_name,description,value,unit
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89,"La Victoria Green Taco Sauce Mild, 12 Ounce (P...",,72.0,Fl Oz
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12,"Salerno Cookies, The Original Butter Cookies, ...",Original Butter Cookies: Classic butter cookie...,32.0,Ounce
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97,"Bear Creek Hearty Soup Bowl, Creamy Chicken wi...",Loaded with hearty long grain wild rice and ve...,11.4,Ounce
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34,Judee’s Blue Cheese Powder 11.25 oz - Gluten-F...,"Add to your favorite appetizers, dips & spread...",11.25,Ounce
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49,"kedem Sherry Cooking Wine, 12.7 Ounce - 12 per...",,12.0,Count
5,9259,"Item Name: Member's Mark Member's Mark, Basil,...",https://m.media-amazon.com/images/I/81nw0HXpCR...,18.50,"Member's Mark Member's Mark, Basil, 6.25 oz","Green Herb, Italian Staple, Great mixed with O...",6.25,ounce
6,191846,Item Name: Goya Foods Sazonador Total Seasonin...,https://m.media-amazon.com/images/I/61dH2Ebkt0...,5.99,"Goya Foods Sazonador Total Seasoning, 30 Ounce...",SAZONADOR TOTAL | Enhance the natural flavors ...,180.0,Ounce
7,222007,Item Name: VineCo Original Series Chilean Sauv...,https://m.media-amazon.com/images/I/71JllaFpxM...,94.00,VineCo Original Series Chilean Sauvignon Blanc...,Chilean Sauvignon Blanc Wine Kit - VineCo Orig...,1.0,Count
8,37614,Item Name: NATURES PATH CEREAL FLK MULTIGRAIN ...,https://m.media-amazon.com/images/I/21O9RftI2v...,35.74,"NATURES PATH CEREAL FLK MULTIGRAIN ORG ECO, 32...",,192.0,Fl Oz
9,238044,Item Name: Mrs. Miller's Seedless Black Raspbe...,https://m.media-amazon.com/images/I/41miQk+RkJ...,31.80,Mrs. Miller's Seedless Black Raspberry Jam 9 O...,Homemade Seedless Black Raspberry Jam made fro...,9.0,Ounce


In [38]:
import re
import string

def clean_text(text):
    if not isinstance(text, str) or text.strip() == "":
        return "other"

    # Remove emojis
    text = re.sub(r'[\U00010000-\U0010ffff]|[\u2600-\u26FF\u2700-\u27BF]', '', text)

    # Remove punctuation and special characters (commas, periods, bullets, etc.)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"[\•\–\—\✓]", "", text)  # extra symbols

    # Remove extra whitespaces
    text = re.sub(r"\s+", " ", text)

    # Convert to lowercase
    text = text.lower().strip()

    return text


In [39]:
df_train['item_name'] = df_train['item_name'].apply(clean_text)
df_train['description'] = df_train['description'].apply(clean_text)

df_test['item_name'] = df_test['item_name'].apply(clean_text)
df_test['description'] = df_test['description'].apply(clean_text)


In [43]:
df_train.head(10)

,sample_id,catalog_content,image_link,price,item_name,description,value,unit
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89,la victoria green taco sauce mild 12 ounce pac...,other,72.0,Fl Oz
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12,salerno cookies the original butter cookies 8 ...,original butter cookies classic butter cookies...,32.0,Ounce
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97,bear creek hearty soup bowl creamy chicken wit...,loaded with hearty long grain wild rice and ve...,11.4,Ounce
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34,judee’s blue cheese powder 1125 oz glutenfree ...,add to your favorite appetizers dips spreads u...,11.25,Ounce
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49,kedem sherry cooking wine 127 ounce 12 per cas...,other,12.0,Count
5,9259,"Item Name: Member's Mark Member's Mark, Basil,...",https://m.media-amazon.com/images/I/81nw0HXpCR...,18.50,members mark members mark basil 625 oz,green herb italian staple great mixed with ore...,6.25,ounce
6,191846,Item Name: Goya Foods Sazonador Total Seasonin...,https://m.media-amazon.com/images/I/61dH2Ebkt0...,5.99,goya foods sazonador total seasoning 30 ounce ...,sazonador total enhance the natural flavors of...,180.0,Ounce
7,222007,Item Name: VineCo Original Series Chilean Sauv...,https://m.media-amazon.com/images/I/71JllaFpxM...,94.00,vineco original series chilean sauvignon blanc...,chilean sauvignon blanc wine kit vineco origin...,1.0,Count
8,37614,Item Name: NATURES PATH CEREAL FLK MULTIGRAIN ...,https://m.media-amazon.com/images/I/21O9RftI2v...,35.74,natures path cereal flk multigrain org eco 32 ...,other,192.0,Fl Oz
9,238044,Item Name: Mrs. Miller's Seedless Black Raspbe...,https://m.media-amazon.com/images/I/41miQk+RkJ...,31.80,mrs millers seedless black raspberry jam 9 oun...,homemade seedless black raspberry jam made fro...,9.0,Ounce


In [44]:
df_test.head(10)

,sample_id,catalog_content,image_link,item_name,description,value,unit
0,100179,Item Name: Rani 14-Spice Eshamaya's Mango Chut...,https://m.media-amazon.com/images/I/71hoAn78AW...,rani 14spice eshamayas mango chutney indian pr...,youll love our 14spice eshamayas mango chutney...,10.5,Ounce
1,245611,Item Name: Natural MILK TEA Flavoring extract ...,https://m.media-amazon.com/images/I/61ex8NHCIj...,natural milk tea flavoring extract by halo pan...,authentic tasting asianinspired natural flavor...,2.0,Fl Oz
2,146263,Item Name: Honey Filled Hard Candy - Bulk Pack...,https://m.media-amazon.com/images/I/61KCM61J8e...,honey filled hard candy bulk pack 2 pounds ind...,honey filled hard candy 2pound bulk pack appro...,32.0,Ounce
3,95658,Item Name: Vlasic Snack'mm's Kosher Dill 16 Oz...,https://m.media-amazon.com/images/I/51Ex6uOH7y...,vlasic snackmms kosher dill 16 oz pack of 2,other,2.0,Count
4,36806,"Item Name: McCormick Culinary Vanilla Extract,...",https://m.media-amazon.com/images/I/71QYlrOMoS...,mccormick culinary vanilla extract 32 fl oz on...,premium ingredients mccormick culinary pure va...,32.0,Fl Oz
5,148239,"Item Name: Snyder's of Hanover Mini Pretzel, 1...",https://m.media-amazon.com/images/I/81ONUU3OAf...,snyders of hanover mini pretzel 16 oz,16 ounce bag of mini pretzels fat free snack,16.0,Ounce
6,92659,Item Name: Oregon Plum Purple 15 oz (Pack of 3...,https://m.media-amazon.com/images/I/519Plng8cp...,oregon plum purple 15 oz pack of 3,oregon fruit products canned fruit fruit choic...,45.0,Ounce
7,3780,"Item Name: Barkman Honey 059640 Honey, Clover,...",https://m.media-amazon.com/images/I/6134TQBo+v...,barkman honey 059640 honey clover 12 oz multi,pure honey does not contain any artificial fla...,1.0,Count
8,196940,Item Name: Against The Grain Gluten Free Origi...,https://m.media-amazon.com/images/I/41bK3vFt5R...,against the grain gluten free original baguett...,other,180.0,oz
9,20472,"Item Name: Nature Valley Granola Bars, Sweet a...",https://m.media-amazon.com/images/I/81NWfSFYXB...,nature valley granola bars sweet and salty nut...,chewy granola bars crunchy almonds and chewy g...,18.0,Ounce


In [41]:
df_train.isnull().sum()

sample_id          0
catalog_content    0
image_link         0
price              0
item_name          0
description        0
value              0
unit               0
dtype: int64

In [45]:
df_test.isnull().sum()

sample_id          0
catalog_content    0
image_link         0
item_name          0
description        0
value              0
unit               0
dtype: int64

In [49]:
df_train['value'].unique()

array(['72.0', '32.0', '11.4', ..., '183.36', '53.66', '3.31'],
      dtype=object)

In [50]:
df_train['unit'].unique()

array(['Fl Oz', 'Ounce', 'Count', 'ounce', 'None', 'Fluid Ounce', 'count',
       'oz', 'Ounces', 'pound', 'fl oz', 'gram', 'grams', 'COUNT',
       'FL Oz', 'lb', 'Each', 'Liters', 'gramm', 'ct', 'Pound', 'Oz',
       'OZ', 'millilitre', 'Jar', 'ounces', 'Fl', 'bottle', 'Bottle',
       'Gram', 'Can', 'Tea Bags', 'Fl oz', 'each', '', 'Pack', 'Piece',
       'fluid ounces', 'gr', 'milliliter', 'mililitro', 'CT', 'FL OZ',
       'pack', 'pounds', 'kg', 'Bag', 'in', 'fl', 'K', 'fluid ounce',
       'sq ft', 'ml', 'Packs', 'box', 'Fl Ounce', 'Pouch', 'Bucket', 'LB',
       'per Box', 'Per Package', 'Comes as a single', 'units', 'CASE',
       'packs', 'BOX', 'Fluid Ounces', 'product', 'Sq Ft', 'per Carton',
       'Foot', 'Grams', 'cm', 'Box', 'unit', 'Paper Cupcake Liners',
       'capsule', 'bottles', 'bag', 'Pounds', 'Ziplock bags',
       'Fluid ounce', 'ltr', 'PACK', 'can', 'Carton', 'Tea bags'],
      dtype=object)

In [51]:
df_train['value'].value_counts()

value
1.0                  6756
16.0                 4016
12.0                 3604
8.0                  2172
24.0                 2076
                     ... 
4.800000000000001       1
10.0178                 1
6.52                    1
44.9                    1
3.31                    1
Name: count, Length: 3107, dtype: int64

In [52]:
df_train['unit'].value_counts()

unit
Ounce                40982
Count                17452
Fl Oz                11082
ounce                 1960
oz                     962
                     ...  
Comes as a single        1
Per Package              1
Packs                    1
Bucket                   1
Tea bags                 1
Name: count, Length: 87, dtype: int64

In [56]:
other_desc_count = (df_train['description'] == "other").sum()
print(other_desc_count)

total_rows = len(df_train)
print(f"Percentage of 'other' descriptions: {other_desc_count/total_rows*100:.2f}%")


13666
Percentage of 'other' descriptions: 18.22%


In [54]:
df_test['unit'].unique()

array(['Ounce', 'Fl Oz', 'Count', 'oz', 'ounce', 'None', 'Oz', 'count',
       'each', 'Fl', 'Box', 'fl oz', 'FL OZ', 'pound', 'packs', 'Pack',
       'FL Oz', 'milliliter', 'Fluid Ounce', '', 'gram', 'Packet',
       'Pound', 'Fl oz', 'product', 'grams', 'lb', 'KIT', 'NA', 'fluid',
       'kg', 'tea bags', 'Tea Bags', 'OZ', 'Liters', 'Packs', 'ct',
       'millilitro', 'BAG', 'Ounces', 'Gram', 'Each', 'Container', 'pac',
       'Milliliters', 'Foot', 'CT', 'fluid ounce', 'Paper Cupcake Liners',
       'COUNT', 'Bag', 'LB', 'ounces', 'millilitre', 'K', 'M', 'Sq Ft',
       'Bottle', 'bottle', 'fluid ounces', 'g', 'PACK', 'Ltr', 'EA',
       'Fluid ounce', 'ml', 'unit', 'SACHET', 'ea', 'Per Package', 'Jar',
       'Tin', 'Liter', 'lbs', 'Pieces', 'Piece', 'Case', 'Fl Ounce',
       'bag', 'Gallon', 'JARS', 'BOX', 'Fluid Ounces', 'gallons',
       'Pounds', 'fl', 'Cou', 'Sugar Substitute', 'Fl OZ', 'Pouch',
       'Capsules', 'st', 'jar', 'Comes as a single dispenser unit with',
       '

In [57]:
other_desc_count = (df_test['description'] == "other").sum()
print(other_desc_count)

total_rows = len(df_test)
print(f"Percentage of 'other' descriptions: {other_desc_count/total_rows*100:.2f}%")


13708
Percentage of 'other' descriptions: 18.28%


In [95]:
other_desc_count = (df_train['item_name'] == "other").sum()
print(other_desc_count)

4


In [63]:
other_desc_count = (df_train['unit'] == "other").sum()
print(other_desc_count)

0


In [65]:
# For train dataset
df_train['description'] = df_train.apply(
    lambda row: row['item_name'] if row['description'] == 'other' else row['description'],
    axis=1
)

df_train['item_name'] = df_train.apply(
    lambda row: row['description'] if row['item_name'] == 'other' else row['item_name'],
    axis=1
)

# For test dataset
df_test['description'] = df_test.apply(
    lambda row: row['item_name'] if row['description'] == 'other' else row['description'],
    axis=1
)

df_test['item_name'] = df_test.apply(
    lambda row: row['description'] if row['item_name'] == 'other' else row['item_name'],
    axis=1
)

# Check how many 'other' are left
print("Train description 'other' count:", (df_train['description'] == 'other').sum())
print("Train item_name 'other' count:", (df_train['item_name'] == 'other').sum())
print("Test description 'other' count:", (df_test['description'] == 'other').sum())
print("Test item_name 'other' count:", (df_test['item_name'] == 'other').sum())


Train description 'other' count: 4
Train item_name 'other' count: 4
Test description 'other' count: 3
Test item_name 'other' count: 3


In [68]:
df_train.head()

,sample_id,catalog_content,image_link,price,item_name,description,value,unit
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89,la victoria green taco sauce mild 12 ounce pac...,la victoria green taco sauce mild 12 ounce pac...,72.0,Fl Oz
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12,salerno cookies the original butter cookies 8 ...,original butter cookies classic butter cookies...,32.0,Ounce
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97,bear creek hearty soup bowl creamy chicken wit...,loaded with hearty long grain wild rice and ve...,11.4,Ounce
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34,judee’s blue cheese powder 1125 oz glutenfree ...,add to your favorite appetizers dips spreads u...,11.25,Ounce
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49,kedem sherry cooking wine 127 ounce 12 per cas...,kedem sherry cooking wine 127 ounce 12 per cas...,12.0,Count


In [74]:
duplicate_rows = df_train[df_train.duplicated()]
print("Number of duplicate rows:", duplicate_rows.shape[0])

Number of duplicate rows: 0


In [77]:
# Convert all units to lowercase
df_train['unit'] = df_train['unit'].str.lower()
df_test['unit'] = df_test['unit'].str.lower()

# Optional: strip extra whitespace
df_train['unit'] = df_train['unit'].str.strip()
df_test['unit'] = df_test['unit'].str.strip()


In [78]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75000 entries, 0 to 74999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   sample_id        75000 non-null  int64  
 1   catalog_content  75000 non-null  object 
 2   image_link       75000 non-null  object 
 3   price            75000 non-null  float64
 4   item_name        75000 non-null  object 
 5   description      75000 non-null  object 
 6   value            75000 non-null  object 
 7   unit             75000 non-null  object 
dtypes: float64(1), int64(1), object(6)
memory usage: 4.6+ MB


In [81]:
df_train.head()

,sample_id,catalog_content,image_link,price,item_name,description,value,unit
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89,la victoria green taco sauce mild 12 ounce pac...,la victoria green taco sauce mild 12 ounce pac...,72.0,fl oz
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12,salerno cookies the original butter cookies 8 ...,original butter cookies classic butter cookies...,32.0,ounce
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97,bear creek hearty soup bowl creamy chicken wit...,loaded with hearty long grain wild rice and ve...,11.4,ounce
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34,judee’s blue cheese powder 1125 oz glutenfree ...,add to your favorite appetizers dips spreads u...,11.25,ounce
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49,kedem sherry cooking wine 127 ounce 12 per cas...,kedem sherry cooking wine 127 ounce 12 per cas...,12.0,count


In [82]:
df_test.head()

,sample_id,catalog_content,image_link,item_name,description,value,unit
0,100179,Item Name: Rani 14-Spice Eshamaya's Mango Chut...,https://m.media-amazon.com/images/I/71hoAn78AW...,rani 14spice eshamayas mango chutney indian pr...,youll love our 14spice eshamayas mango chutney...,10.5,ounce
1,245611,Item Name: Natural MILK TEA Flavoring extract ...,https://m.media-amazon.com/images/I/61ex8NHCIj...,natural milk tea flavoring extract by halo pan...,authentic tasting asianinspired natural flavor...,2.0,fl oz
2,146263,Item Name: Honey Filled Hard Candy - Bulk Pack...,https://m.media-amazon.com/images/I/61KCM61J8e...,honey filled hard candy bulk pack 2 pounds ind...,honey filled hard candy 2pound bulk pack appro...,32.0,ounce
3,95658,Item Name: Vlasic Snack'mm's Kosher Dill 16 Oz...,https://m.media-amazon.com/images/I/51Ex6uOH7y...,vlasic snackmms kosher dill 16 oz pack of 2,vlasic snackmms kosher dill 16 oz pack of 2,2.0,count
4,36806,"Item Name: McCormick Culinary Vanilla Extract,...",https://m.media-amazon.com/images/I/71QYlrOMoS...,mccormick culinary vanilla extract 32 fl oz on...,premium ingredients mccormick culinary pure va...,32.0,fl oz


In [80]:
df_train['unit'].unique()

array(['fl oz', 'ounce', 'count', 'none', 'fluid ounce', 'oz', 'ounces',
       'pound', 'gram', 'grams', 'lb', 'each', 'liters', 'gramm', 'ct',
       'millilitre', 'jar', 'fl', 'bottle', 'can', 'tea bags', '', 'pack',
       'piece', 'fluid ounces', 'gr', 'milliliter', 'mililitro', 'pounds',
       'kg', 'bag', 'in', 'k', 'sq ft', 'ml', 'packs', 'box', 'fl ounce',
       'pouch', 'bucket', 'per box', 'per package', 'comes as a single',
       'units', 'case', 'product', 'per carton', 'foot', 'cm', 'unit',
       'paper cupcake liners', 'capsule', 'bottles', 'ziplock bags',
       'ltr', 'carton'], dtype=object)

In [83]:
df_test['unit'].unique()

array(['ounce', 'fl oz', 'count', 'oz', 'none', 'each', 'fl', 'box',
       'pound', 'packs', 'pack', 'milliliter', 'fluid ounce', '', 'gram',
       'packet', 'product', 'grams', 'lb', 'kit', 'na', 'fluid', 'kg',
       'tea bags', 'liters', 'ct', 'millilitro', 'bag', 'ounces',
       'container', 'pac', 'milliliters', 'foot', 'paper cupcake liners',
       'millilitre', 'k', 'm', 'sq ft', 'bottle', 'fluid ounces', 'g',
       'ltr', 'ea', 'ml', 'unit', 'sachet', 'per package', 'jar', 'tin',
       'liter', 'lbs', 'pieces', 'piece', 'case', 'fl ounce', 'gallon',
       'jars', 'gallons', 'pounds', 'cou', 'sugar substitute', 'pouch',
       'capsules', 'st', 'comes as a single dispenser unit with',
       'mililitro'], dtype=object)

In [84]:
df_train['unit'].value_counts()

unit
ounce                   42942
count                   18209
fl oz                   11257
oz                       1064
none                      940
pound                     202
ct                         49
gram                       39
fluid ounce                39
lb                         34
each                       26
fl                         23
ounces                     20
fluid ounces               14
                           12
packs                      12
millilitre                  9
pack                        9
bottle                      9
liters                      7
kg                          6
can                         6
bag                         6
gramm                       6
pounds                      4
jar                         4
grams                       4
piece                       3
per carton                  3
k                           3
milliliter                  3
box                         3
gr                          2
pouch

In [85]:
df_test['unit'].value_counts()

unit
ounce         42884
count         18188
fl oz         11348
oz             1019
none           1011
              ...  
fluid             1
m                 1
millilitro        1
container         1
mililitro         1
Name: count, Length: 66, dtype: int64

In [89]:
unit_mapping = {
    'oz': 'oz', 'ounce': 'oz', 'ounces': 'oz',
    'fl oz': 'fl_oz', 'fluid ounce': 'fl_oz', 'fluid ounces': 'fl_oz', 'fl ounce': 'fl_oz', 'fl': 'fl_oz',
    'lb': 'lb', 'pound': 'lb', 'pounds': 'lb', 'lbs': 'lb',
    'g': 'g', 'gram': 'g', 'grams': 'g', 'gramm': 'g', 'gr': 'g', 'kg': 'kg',
    'ml': 'ml', 'milliliter': 'ml', 'milliliters': 'ml', 'millilitre': 'ml', 'mililitro': 'ml',
    'l': 'l', 'liter': 'l', 'liters': 'l', 'ltr': 'l',
    'gallon': 'gallon', 'gallons': 'gallon',
    'count': 'count', 'ct': 'count', 'each': 'count', 'unit': 'count', 'units': 'count',
    'pack': 'count', 'packs': 'count', 'piece': 'count', 'pieces': 'count', 'cou': 'count',
    'bottle': 'bottle', 'bottles': 'bottle', 'jar': 'jar', 'jars': 'jar', 'pouch': 'pouch',
    'cm': 'cm', 'in': 'inch', 'inch': 'inch', 'foot': 'foot', 'sq ft': 'sq_ft',
    '': 'unknown', 'none': 'unknown', 'na': 'unknown', 'product': 'unknown'
}

# Apply mapping
df_train['unit'] = df_train['unit'].str.lower().str.strip()
df_test['unit'] = df_test['unit'].str.lower().str.strip()

df_train['unit'] = df_train['unit'].map(unit_mapping).fillna(df_train['unit'])
df_test['unit'] = df_test['unit'].map(unit_mapping).fillna(df_test['unit'])

In [90]:
df_train['unit'].unique()

array(['fl_oz', 'oz', 'count', 'unknown', 'lb', 'g', 'l', 'ml', 'jar',
       'bottle', 'can', 'tea bags', 'kg', 'bag', 'inch', 'k', 'sq_ft',
       'box', 'pouch', 'bucket', 'per box', 'per package',
       'comes as a single', 'case', 'per carton', 'foot', 'cm',
       'paper cupcake liners', 'capsule', 'ziplock bags', 'carton'],
      dtype=object)

In [91]:
df_test['unit'].unique()

array(['oz', 'fl_oz', 'count', 'unknown', 'box', 'lb', 'ml', 'g',
       'packet', 'kit', 'fluid', 'kg', 'tea bags', 'l', 'millilitro',
       'bag', 'container', 'pac', 'foot', 'paper cupcake liners', 'k',
       'm', 'sq_ft', 'bottle', 'ea', 'sachet', 'per package', 'jar',
       'tin', 'case', 'gallon', 'sugar substitute', 'pouch', 'capsules',
       'st', 'comes as a single dispenser unit with'], dtype=object)

In [93]:
df_train.head(10)

,sample_id,catalog_content,image_link,price,item_name,description,value,unit
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89,la victoria green taco sauce mild 12 ounce pac...,la victoria green taco sauce mild 12 ounce pac...,72.0,fl_oz
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12,salerno cookies the original butter cookies 8 ...,original butter cookies classic butter cookies...,32.0,oz
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97,bear creek hearty soup bowl creamy chicken wit...,loaded with hearty long grain wild rice and ve...,11.4,oz
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34,judee’s blue cheese powder 1125 oz glutenfree ...,add to your favorite appetizers dips spreads u...,11.25,oz
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49,kedem sherry cooking wine 127 ounce 12 per cas...,kedem sherry cooking wine 127 ounce 12 per cas...,12.0,count
5,9259,"Item Name: Member's Mark Member's Mark, Basil,...",https://m.media-amazon.com/images/I/81nw0HXpCR...,18.50,members mark members mark basil 625 oz,green herb italian staple great mixed with ore...,6.25,oz
6,191846,Item Name: Goya Foods Sazonador Total Seasonin...,https://m.media-amazon.com/images/I/61dH2Ebkt0...,5.99,goya foods sazonador total seasoning 30 ounce ...,sazonador total enhance the natural flavors of...,180.0,oz
7,222007,Item Name: VineCo Original Series Chilean Sauv...,https://m.media-amazon.com/images/I/71JllaFpxM...,94.00,vineco original series chilean sauvignon blanc...,chilean sauvignon blanc wine kit vineco origin...,1.0,count
8,37614,Item Name: NATURES PATH CEREAL FLK MULTIGRAIN ...,https://m.media-amazon.com/images/I/21O9RftI2v...,35.74,natures path cereal flk multigrain org eco 32 ...,natures path cereal flk multigrain org eco 32 ...,192.0,fl_oz
9,238044,Item Name: Mrs. Miller's Seedless Black Raspbe...,https://m.media-amazon.com/images/I/41miQk+RkJ...,31.80,mrs millers seedless black raspberry jam 9 oun...,homemade seedless black raspberry jam made fro...,9.0,oz


In [98]:
df_train['unit'].value_counts()

unit
oz                      44026
count                   18310
fl_oz                   11334
unknown                   954
lb                        240
g                          51
ml                         15
bottle                     10
l                           8
can                         6
kg                          6
bag                         6
jar                         4
per carton                  3
box                         3
k                           3
sq_ft                       2
pouch                       2
per box                     2
tea bags                    2
foot                        2
paper cupcake liners        2
inch                        1
bucket                      1
per package                 1
comes as a single           1
case                        1
cm                          1
capsule                     1
ziplock bags                1
carton                      1
Name: count, dtype: int64

In [99]:
df_test['unit'].value_counts()

unit
oz                                       43916
count                                    18291
fl_oz                                    11408
unknown                                   1031
lb                                         221
g                                           36
ml                                          18
l                                            9
bottle                                       9
bag                                          7
box                                          5
jar                                          5
foot                                         4
tea bags                                     4
paper cupcake liners                         3
gallon                                       3
packet                                       3
k                                            3
ea                                           3
sq_ft                                        2
per package                                  2
pac     

In [101]:
df_train['unit'] = df_train['unit'].replace('k', 'kg')
df_test['unit'] = df_test['unit'].replace('k', 'kg')

df_train['unit'] = df_train['unit'].replace('millilitro', 'ml')
df_test['unit'] = df_test['unit'].replace('millilitro', 'ml')


rare_train = df_train['unit'].value_counts()[df_train['unit'].value_counts() < 8].index
rare_test = df_test['unit'].value_counts()[df_test['unit'].value_counts() < 8].index

df_train['unit'] = df_train['unit'].apply(lambda x: 'na' if x in rare_train else x)
df_test['unit'] = df_test['unit'].apply(lambda x: 'na' if x in rare_test else x)

In [102]:
df_train['unit'].value_counts()

unit
oz         44026
count      18310
fl_oz      11334
unknown      954
lb           240
g             51
na            43
ml            15
bottle        10
kg             9
l              8
Name: count, dtype: int64

In [103]:
df_test['unit'].value_counts()

unit
oz         43916
count      18291
fl_oz      11408
unknown     1031
lb           221
na            60
g             36
ml            19
l              9
bottle         9
Name: count, dtype: int64

In [106]:
df_train['unit'] = df_train['unit'].replace({'unknown': 'na'})
df_test['unit'] = df_test['unit'].replace({'unknown': 'na'})

df_train['unit'] = df_train['unit'].str.lower().str.strip()
df_test['unit'] = df_test['unit'].str.lower().str.strip()

In [107]:
df_train['unit'].value_counts()

unit
oz        44026
count     18310
fl_oz     11334
na          997
lb          240
g            51
ml           15
bottle       10
kg            9
l             8
Name: count, dtype: int64

In [108]:
df_test['unit'].value_counts()

unit
oz        43916
count     18291
fl_oz     11408
na         1091
lb          221
g            36
ml           19
l             9
bottle        9
Name: count, dtype: int64

In [110]:
# Map 'k' to 'kg' in test dataset
df_test['unit'] = df_test['unit'].replace('k', 'kg')

# Optional: lowercase and strip
df_test['unit'] = df_test['unit'].str.lower().str.strip()

# Check if 'kg' exists, if not, add a placeholder row
if 'kg' not in df_test['unit'].unique():
    df_test.loc[len(df_test)] = {
        'item_name': 'placeholder',
        'description': 'placeholder',
        'value': 0,
        'unit': 'kg'
    }

# Verify
print(df_test['unit'].value_counts())


unit
oz        43916
count     18291
fl_oz     11408
na         1091
lb          221
g            36
ml           19
l             9
bottle        9
kg            1
Name: count, dtype: int64


In [112]:
df_train['unit'] = df_train['unit'].replace('bottle', 'count')
df_test['unit'] = df_test['unit'].replace('bottle', 'count')

In [113]:
df_train['unit'].value_counts()

unit
oz       44026
count    18320
fl_oz    11334
na         997
lb         240
g           51
ml          15
kg           9
l            8
Name: count, dtype: int64

In [114]:
df_test['unit'].value_counts()

unit
oz       43916
count    18300
fl_oz    11408
na        1091
lb         221
g           36
ml          19
l            9
kg           1
Name: count, dtype: int64

In [118]:
df_train.sample(10)

,sample_id,catalog_content,image_link,price,item_name,description,value,unit
29325,202947,Item Name: José Andrés Foods Mussels In Escabe...,https://m.media-amazon.com/images/I/617XUJjk4g...,161.98,josé andrés foods mussels in escabeche 423 oz ...,plump mussels served in the galician style har...,49.92,oz
57434,46088,Item Name: THAI KITCHEN SAUCE DPPNG CHILI RED ...,https://m.media-amazon.com/images/I/71Fj15VbEY...,5.61,thai kitchen sauce dppng chili red swt 7 oz bu...,designed to taste great please always check th...,42.0,fl_oz
34852,191968,"Item Name: Carniceria Compare, Refried Black B...",https://m.media-amazon.com/images/I/71Tu9MYfqN...,2.69,carniceria compare refried black beans 15 ounce,good source of protein cholesterol free terrif...,15.0,oz
38288,49272,"Item Name: CELTIC SEA SALT Reseal Bag, Flower ...",https://m.media-amazon.com/images/I/81WX3lbfpz...,15.49,celtic sea salt reseal bag flower of ocean,natural ingredients quality tested and approved,4.0,oz
73414,158055,"Item Name: Tate's Bake Shop Cookies, Chocolate...",https://m.media-amazon.com/images/I/81311mMOGO...,5.49,tates bake shop cookies chocolate chip 7 oz,one 7 oz bag of tates bake shop chocolate chip...,7.0,oz
19642,221706,Item Name: Lemon Perfect 4-Flavor Variety Pack...,https://m.media-amazon.com/images/I/81MyK-m1nu...,33.99,lemon perfect 4flavor variety pack hydrating f...,full of flavor water never tasted this good sq...,182.4,fl_oz
22061,119369,Item Name: REESE'S Assorted Milk Chocolate Pea...,https://m.media-amazon.com/images/I/81WV3vpobw...,13.29,reeses assorted milk chocolate peanut butter s...,contains one 1 96ounce variety bag of reeses a...,9.6,oz
2931,126375,Item Name: Original Cadbury Crunchie Chocolate...,https://m.media-amazon.com/images/I/71H+9-KR2o...,14.50,original cadbury crunchie chocolate bar pack c...,a golden honeycombed centre surrounded by deli...,1.69315,oz
70530,71844,"Item Name: Red Rooster Coffee, Organic Funky C...",https://m.media-amazon.com/images/I/61eICJLgHN...,20.00,red rooster coffee organic funky chicken coffe...,craft roasted with purpose every bag of red ro...,12.0,oz
26788,193853,Item Name: Hershey Miniatures Assorted Flavore...,https://m.media-amazon.com/images/I/71m6HScX9u...,8.49,hershey miniatures assorted flavored halloween...,contains one 1 2121ounce 75piece variety bag o...,43.2,oz


In [120]:
df_test.sample(10)

,sample_id,catalog_content,image_link,item_name,description,value,unit
71440,137679.0,"Item Name: Castleberrys Pork in BBQ Sauce, Car...",https://m.media-amazon.com/images/I/61eJz4rj1q...,castleberrys pork in bbq sauce carolina recipe...,made with a premium blend of the highest quali...,10.5,fl_oz
22725,271470.0,"Item Name: Mentos Chewy Mint Candy Roll, Rainb...",https://m.media-amazon.com/images/I/51F6a3st47...,mentos chewy mint candy roll rainbow non melti...,includes mentos chewy candy rolls flavored che...,1.32,oz
58419,68863.0,Item Name: Gourmet Blue Raspberry Flavored Pop...,https://m.media-amazon.com/images/I/91W8O6SqlI...,gourmet blue raspberry flavored popcorn by its...,exciting bluecolored blue popcorn delicious ra...,80.0,oz
27565,278760.0,Item Name: Navitas Organics Goji Berries – For...,https://m.media-amazon.com/images/I/71qr0tn9eK...,navitas organics goji berries for smoothies tr...,vitality longevity an ancient superfood goji b...,16.0,oz
73578,255667.0,Item Name: Pillsbury Radiant Red Vanilla Funfe...,https://m.media-amazon.com/images/I/81DaSmVHoP...,pillsbury radiant red vanilla funfetti frostin...,includes candy bits pack of 2 tubs,31.2,oz
28269,105920.0,Item Name: Fisher Oven Roasted Never Fried Who...,https://m.media-amazon.com/images/I/81kx859L4o...,fisher oven roasted never fried whole cashews ...,convenient snacking ─ contains 1 24ounce reusa...,24.0,oz
71299,16737.0,"Item Name: Keebler Fudge Stripes Cookies, Coco...",https://m.media-amazon.com/images/I/51aAOyatMJ...,keebler fudge stripes cookies coconut dreams f...,coconut dreams flavored cookies soft and chewy...,17.0,oz
23604,35936.0,Item Name: Idahoan Roasted Garlic Mashed Potat...,https://m.media-amazon.com/images/I/21PshHFU6L...,idahoan roasted garlic mashed potatoes 4 oz pa...,made with 100 real idaho potatoes perfectly bl...,48.0,fl_oz
20291,257786.0,Item Name: Betty Crocker Hamburger Helper Baco...,https://m.media-amazon.com/images/I/81mfQZNIy0...,betty crocker hamburger helper bacon cheesebur...,big time flavors combine in this bacon and che...,5.1,oz
43181,79471.0,Item Name: Mrs. Dash CHICKEN GRILLING BLEND Sa...,https://m.media-amazon.com/images/I/61u9EmqYa0...,mrs dash chicken grilling blend saltfree seaso...,mrs dash chicken grilling blend saltfree seaso...,7.5,oz
